# Module 2 — Crawl & Trích Xuất Văn Bản (PDF Text & Web HTML)

Notebook này thực hiện trích xuất dữ liệu từ các tài liệu số hóa đã chuẩn hóa:
- **Tự động kéo dữ liệu:** Tự động phát hiện và kéo tài liệu từ GitHub repo (`https://github.com/caoban123/CV01.git`) nếu chưa có sẵn local.
- **Tự động phân loại:** Soi xét PDF để nhận diện **PDF_TEXT** (trích xuất text layer trực tiếp) hay **PDF_SCAN** (ảnh scan cần OCR).
- **Tự động gán mã:** Gán mã đề tài `HVB_01`, mã tác phẩm, document ID, tính mã SHA-256 provenance theo hợp đồng chuẩn `plan.md`.

In [ ]:
# 1. Cài đặt môi trường & Nạp thư viện
import os
import sys
from pathlib import Path

# Tự động xác định thư mục gốc của repository
REPO_ROOT = Path('.').resolve()
while REPO_ROOT.name and not (REPO_ROOT / 'Crawl').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Crawl.utils.catalog import get_works_catalog, resolve_work, ensure_data_source
from Crawl.utils.id_generator import generate_project_id, generate_document_id, compute_sha256, build_sidecar_metadata
from Crawl.extractors.html_extractor import extract_from_html
from Crawl.extractors.pdf_text_extractor import extract_from_pdf_text
from OCR.utils.pdf_detector import inspect_pdf

# Tự động đảm bảo nguồn tài liệu có sẵn (kéo từ GitHub nếu chạy trên Colab)
source_dir = ensure_data_source(base_dir=str(REPO_ROOT / 'data' / 'source_materials'))
print('✓ Hệ thống nạp thư viện và kiểm tra dữ liệu thành công!')

In [ ]:
# 2. Danh mục tác phẩm & Lựa chọn tài liệu
catalog = get_works_catalog()
print('=== DANH MỤC CÁC TÁC PHẨM CÓ SẴN ===')
for idx, (key, info) in enumerate(catalog.items(), 1):
    print(f"[{idx}] {key:15s} | {info['name']} ({info['author']})")

# Chọn tác phẩm cần trích xuất (HVNTDDC, LTHCLC_Q20_25, LTHCLC_Q46_49, VIET_SU_YEU, DVDDTB)
SELECTED_WORK_KEY = 'LTHCLC_Q20_25'
LANGUAGE = 'viet'  # 'han' hoặc 'viet'
START_PAGE = 1    # Trang bắt đầu
END_PAGE = 15     # Trang kết thúc mẫu (hoặc đặt None để bóc tách toàn bộ tài liệu)

work_info = resolve_work(SELECTED_WORK_KEY, base_dir=str(source_dir))
print(f"\nĐang chọn: {work_info['name']} ({work_info['ma_tac_pham']}) — Ngôn ngữ: {LANGUAGE.upper()}")
target_file = work_info['han_abs_path'] if LANGUAGE == 'han' else work_info['viet_abs_path']
print(f"Đường dẫn file: {target_file}")

In [ ]:
# 3. Tự động kiểm tra PDF Text vs PDF Scan
if not target_file or not Path(target_file).exists():
    print(f'[LƯU Ý] Chưa thấy file lớn tại {target_file}. Dùng file mẫu kiểm thử fixture...')
    target_file = str(REPO_ROOT / 'tests' / 'fixtures' / 'sample_text.pdf')

pdf_info = inspect_pdf(target_file, sample_pages=5)
print('=== KẾT QUẢ TỰ ĐỘNG PHÂN TÍCH PDF ===')
print(f"Loại tài liệu:     {pdf_info['pdf_type']}")
print(f"Tổng số trang:     {pdf_info['total_pages']}")
print(f"Ký tự mẫu/trang:   {pdf_info['avg_chars_per_page']}")
print(f"Khuyến nghị:       {pdf_info['recommended_engine']}")
print(f"Nhận định:         {pdf_info['reason']}")

if pdf_info['pdf_type'] == 'PDF_SCAN':
    print('\n[CẢNH BÁO] Đây là file scan ảnh. Khuyến nghị chạy bằng Module OCR (OCR/notebooks/01_OCR_Han.ipynb) để có độ chính xác cao nhất.')

In [ ]:
# 4. Trích xuất văn bản & Tự động gán mã ID chuẩn
output_dir = REPO_ROOT / 'data' / 'crawled'
output_dir.mkdir(parents=True, exist_ok=True)

if pdf_info['pdf_type'] == 'PDF_TEXT':
    result = extract_from_pdf_text(target_file, start_page=START_PAGE, end_page=END_PAGE)
    extracted_text = result['text']
else:
    result = extract_from_pdf_text(target_file, start_page=START_PAGE, end_page=END_PAGE)
    extracted_text = result['text']

# Tính mã SHA-256 provenance và sinh Document ID
source_sha256 = compute_sha256(target_file)
doc_id = generate_document_id(work_info['ma_tac_pham'], LANGUAGE, source_sha256)
project_id = generate_project_id(work_info['ma_de_tai'], work_info['ma_tac_pham'])

lines = [l.strip() for l in extracted_text.splitlines() if l.strip()]
print(f'\n✓ Đã trích xuất: {len(extracted_text)} ký tự ({len(lines)} câu/dòng)')
print(f'Document ID: {doc_id}')
print(f'Project ID:  {project_id}')
print(f'SHA-256:     {source_sha256[:16]}...')

print('\n--- 5 DÒNG ĐẦU TIÊN ---')
for line in lines[:5]:
    print('>', line)

In [ ]:
# 5. Lưu kết quả ra file chuẩn hóa phục vụ Web Studio
import json

out_txt = output_dir / f"{LANGUAGE}.txt"
out_meta = output_dir / f"{LANGUAGE}_metadata.json"

with open(out_txt, 'w', encoding='utf-8', newline='\n') as f:
    f.write(extracted_text)

meta = build_sidecar_metadata(
    ma_de_tai=work_info['ma_de_tai'],
    ma_tac_pham=work_info['ma_tac_pham'],
    tac_pham_name=work_info['name'],
    language=LANGUAGE,
    source_file_path=target_file,
    total_segments=len(lines),
    total_characters=len(extracted_text),
    extractor_type='pdf_text_extractor' if pdf_info['pdf_type'] == 'PDF_TEXT' else 'pdf_scan_fallback',
    hardware='cpu'
)

with open(out_meta, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f'✓ Đã lưu file text sạch: {out_txt}')
print(f'✓ Đã lưu metadata:   {out_meta}')
print('\nSẵn sàng tải lên Han–Viet Alignment Studio tại http://127.0.0.1:8000!')